# Proyecto 1 — Analítica de Datos
## Modelamiento de la precipitación semanal en el Valle del Cauca

**Universidad Autónoma de Occidente** · Ingeniería de Datos e Inteligencia Artificial  
Analítica de Datos 2026-2S · Prof. Johann A. Ospina  
César Armando Reyes Oliveros — 2236379 · Yesenia Díaz — 2231783 · Juan Pablo Maya

---

### Para qué sirve este cuaderno

Registra **qué** hacemos, **por qué**, en qué **material de clase** se apoya cada
decisión y qué **resultado** dio. Sirve para que el grupo siga el hilo y para
preparar la sustentación.

> ⚠️ **No es el entregable.** El enunciado exige **PDF de máximo 10 páginas** más
> **un único `.R`**, y prohíbe R Markdown o solo-Colab. El código canónico vive en
> **`R/proyecto1.R`**; aquí solo se explica.


---
## Cómo trabajar en RStudio

🔴 **Empieza SIEMPRE cada sesión con esto:**


In [ ]:
source("R/proyecto1.R")


Corre el análisis completo (~30 s) y deja en memoria todo lo que necesitas:

| Objeto | Qué es |
|---|---|
| `datos` | las 70 estaciones, 9 columnas |
| `borde` | polígono del departamento |
| `precip`, `r_alt`, `mascara` | rasters recortados al área |
| `modelo_tendencia` | el `lm()` ajustado |
| `figura()`, `mapa_base()`, `esquina_leyenda()` | helpers de dibujo |
| `col_banda`, `pal_lon`, `cortes_x`, `lat0`, `lon0` | auxiliares de las figuras |

> ⚠️ **Este fue el error del 11-sep.** Se intentó correr los comandos de las figuras
> sin haber ejecutado antes el modelo de tendencia. `datos$residual` no existía y R
> respondió `non-numeric argument to mathematical function` — un mensaje que no dice
> nada sobre la causa real. Con `source()` primero, no vuelve a pasar.


---
## Paso 0 — El encargo, el molde y los datos

### Qué pide el enunciado

Modelar la precipitación semanal en el Valle del Cauca en función de covariables
ambientales y **evaluar el papel de la correlación espacial**. Cuatro bloques:
EDA → Modelación → Validación → Predicción, justificando cada uno paso a paso.

La restricción que manda sobre todo:

> *"Utilizar únicamente los scripts, procedimientos y metodologías trabajados en
> clase. Librerías adicionales permitidas solo para: elaboración de gráficos y carga
> de mapas o archivos de Excel."*

### El molde

**`Ejemplo3_Geoestadística.R`** es el más cercano: CHIRPS semanal, Valle del Cauca,
borde GADM, kriging. Es el mismo problema. **`Ejemplo4_Geoestadística.R`** tiene el
flujo completo mejor ordenado. Ambos en `material_clase/Practice Geostatistics/`.

El profesor dejó una nota en la cabecera del Ejemplo3 que conviene tener presente:

> *"las funciones están con la teoría vista en clase son correctas, sin embargo, no
> quiere decir que los resultados sean apropiados. **Usted debe leer y ajustar lo
> necesario** para que los resultados sean apropiados"*

| Sección del molde | Qué hace | Nuestro paso |
|---|---|---|
| Cargar imágenes | `rast()` | 1 |
| Borde del valle | `as.polygons()` sobre la máscara | 2.1 |
| Puntos de muestreo | `spatSample(method = "random")` | 2.2 |
| Proyección | grados → km, coseno de la latitud | 2.2 |
| EDA espacial | histogramas, posting, dispersión | 3 |
| Extraer tendencia | `lm(z ~ x_km + y_km)` | 4 |
| Distancias | `dist()` | 5 |
| Semivariograma | nube + bins con `tapply()` | 6 |
| Ajuste de modelo | `optim()` L-BFGS-B multi-arranque | 6 |
| Predicción | sistema kriging con `solve()` | 7 |
| Validación cruzada | LOOCV a mano | 8 |

### Librerías

✅ Con precedente: `terra`, `sp`, `gstat`, `geodata`, `chirps`, `ggplot2`, `gridExtra`,
`geostan`, `openxlsx`, `dplyr`, y base R (`lm`, `optim`, `solve`, `dist`, `tapply`, `cut`).

❌ Sin precedente: `automap`, `spdep`, `sf`, `caret`, `randomForest`, `mgcv`, `tmap`,
`leaflet`, el paquete viejo `raster`, y `terra::interpIDW`.

**Decisión: solo `terra` + base R.** Es el subconjunto más defendible y es lo que usa
el molde.

### Verificación de los datos

```
md5  d1bd258c058004a7877617817027b60a   datos_proyecto_1.zip   respaldo == repo ✅
md5  813557728ea4ceee9b5977e37bc68789   enunciado PDF          idéntico      ✅
CRC  229 de 229 .tif extraídos coinciden con el zip                          ✅
```


---
# Paso 1 — Auditoría de los datos

**Qué.** Verificar geometría, valores centinela y cobertura real antes de modelar.

**Por qué.** El enunciado afirma que las capas son *"directamente comparables píxel
a píxel"*. Cierto en geometría, **falso en cobertura**.

**Material.** `Ejemplo4` (carga con `rast`), `carpeta/Script_Class5.R` (`terra`).


## 1.1 Geometría — se cumple

```
              filas cols bandas  res   xmin  xmax  ymin  ymax
Precipitacion    39   37     52 0.05 -77.55 -75.7  3.05  5.00
Temperatura      39   37     52 0.05 -77.55 -75.7  3.05  5.00
Radiacion        39   37     52 0.05 -77.55 -75.7  3.05  5.00
Altitud          39   37      1 0.05 -77.55 -75.7  3.05  5.00
```

39 × 37 = **1 443 píxeles** de 0,05° (≈ 5,6 km), EPSG:4326. Las 52 bandas son las
semanas ISO.


## 1.2 Valores centinela

CHIRPS codifica "sin dato" como un negativo enorme, no como `NA`. La lluvia no puede
ser negativa; si no se filtra, **todo lo que sigue es basura**.

```
Mínimo crudo:     -76867.3
Celdas negativas:      104
Mínimo tras limpiar:  2.77 mm   ← valor físico
```


In [ ]:
r_precip[r_precip < 0] <- NA


## 1.3 La máscara del área de estudio

El rectángulo de 1 443 píxeles **no** es el departamento: incluye océano Pacífico y
territorio vecino. Sin máscara no hay denominador honesto.

> 🔴 **El error que cometimos.** La primera versión midió cobertura contra las 1 443
> celdas del rectángulo y dio `Altitud → 47,7 % → SE DESCARTA`, descartando la mejor
> covariable.
> 
> ```
> altitud / rectángulo   = 688/1443 = 47,7 %  →  "media cobertura"
> altitud / departamento =  688/688 =  100 %  →  "cobertura total"
> ```
> 
> **Define la máscara antes de calcular cualquier porcentaje.**


In [ ]:
mascara <- !is.na(r_alt) & !is.na(r_precip[[SEMANA]])
N_VALLE <- sum(values(mascara), na.rm = TRUE)   # 688 celdas


## 1.4 Cobertura efectiva — el hallazgo que define el modelo

| Variable | Celdas | % del área | Valores distintos | |
|---|---|---|---|---|
| Precipitación | 688 | 100,0 % | 688 | ✅ |
| Altitud | 688 | 100,0 % | 688 | ✅ |
| Temperatura | 123 | 17,9 % | 56 | ❌ |
| **Radiación** | **28** | **4,1 %** | **3** | ❌ |

**Por qué.** CHIRPS es nativo 0,05°; NASA POWER es nativo **0,5°**, diez veces más
grueso. Al llevarlo a la grilla fina solo sobrevivieron los centros originales. El
enunciado dice "remuestreadas a la resolución de CHIRPS", pero eso fue reproyección
de rejilla, **no relleno**.

**Por qué importa.** La radiación, dentro del Valle, son **3 números**. No es un
gradiente: es un escalón costa/valle/cordillera. Meterla al `lm()` como continua le
atribuye a "radiación" lo que es **posición geográfica**, que ya entra como `x_km`.

**Decisión.** Temperatura y radiación **fuera**. El modelo usa **altitud +
coordenadas**, las tres con 100 % de cobertura y cero dato inventado.

Esto además evita `terra::interpIDW`, que es como se rellenarían esos huecos y que
**no aparece en ningún script de clase** (verificado con `grep` sobre `material_clase/`).


## 1.5 Elección de la semana — con evidencia

"Octubre es lluvioso" es razonable, pero el dato decide. Se promedian las 52 semanas
y se toma el máximo.

```
Semana más lluviosa (climatología 2010-2025): 44 (91,9 mm)
Semana más seca:                               3 (28,7 mm)
```

La **44** (principios de noviembre), no la 42.


In [ ]:
media_semanal <- sapply(1:nlyr(r_precip),
                        function(k) mean(values(r_precip[[k]]), na.rm = TRUE))
SEMANA <- which.max(media_semanal)   # -> 44


![Ciclo anual](resultados/fig01_ciclo_anual.png)

**Interpretación.** Régimen **bimodal** andino: dos picos al año (abril-mayo y
octubre-noviembre) por el doble paso de la Zona de Convergencia Intertropical. El
mínimo de la semana 3 es el veranillo de enero. La semana elegida tiene más del
triple de lluvia que la más seca, así que la señal espacial será fuerte.


---
# Paso 2 — Puntos de muestreo

## 2.1 El borde real del departamento

**Qué.** Convertir la máscara en polígono. **Por qué.** Los puntos hay que sortearlos
*dentro* del departamento; sobre el rectángulo caerían en el Pacífico.

**Material.** `Ejemplo4_Geoestadística.R`, líneas 21-24.


In [ ]:
borde <- as.polygons(mascara, dissolve = TRUE)
borde <- borde[borde[[1]] == 1, ]      # <- la línea que la gente olvida

# CHIRPS (787 celdas) desborda el departamento (688) y cubre mar abierto.
precip <- mask(r_precip[[SEMANA]], mascara, maskvalues = c(FALSE, NA))


### La línea que la gente olvida

`as.polygons()` sobre un raster lógico devuelve **dos** polígonos: el de los `TRUE`
(atributo 1) y el de los `FALSE` (atributo 0). Sin `borde[borde[[1]] == 1, ]` el
"borde" incluye el océano.

### Verificación

```
Polígonos devueltos por as.polygons(): 2
Área del borde conservado:        21 125 km²   (real: 22 140 km² → 95 %)
Celdas de CHIRPS antes del recorte:  787
Celdas tras recortar al área:        688
```


![Borde del Valle](resultados/fig02_borde_valle.png)

**Interpretación — el hallazgo central del EDA.**

| Zona | Lluvia semana 44 |
|---|---|
| Vertiente pacífica (Buenaventura, verde) | **200+ mm** |
| Valle interandino (blanco) | **30-50 mm** |

Factor de **5 a 7×** en ~150 km. Efecto orográfico: la humedad del Pacífico choca
contra la cordillera Occidental, descarga en la vertiente y llega seca al valle.

**Consecuencia:** buena parte de la varianza es **tendencia de gran escala en la
longitud**, no estructura aleatoria. Por eso el modelo se parte en dos: una tendencia
determinística y un residuo espacialmente correlacionado que interpola el kriging.


## 2.2 Sorteo de las estaciones

**Qué.** Sortear 70 celdas dentro del borde y proyectar a kilómetros.

**Por qué no usar las 688 celdas:**

1. **Costo.** El sistema kriging invierte una matriz (n+1)×(n+1) por cada punto
   predicho y por cada iteración del LOOCV. Con n = 688 es inviable.
2. **Sentido.** Un semivariograma modela un proceso muestreado en **estaciones**. Con
   las 688 celdas ya tienes el mapa y no hay nada que interpolar.

**Material.** `Ejemplo4` sortea 80 puntos; `Ejemplo3` sortea 40. El muestreo aleatorio
de la grilla **es el idioma del profesor**, no un atajo.

> ⚠️ **Esto hay que escribirlo en el informe**, porque es la crítica obvia en la
> sustentación: las 688 celdas vienen de un producto grillado (CHIRPS), no de
> estaciones reales. El muestreo **simula** una red de observación para poder evaluar
> el kriging contra un valor conocido. El R² de validación mide **capacidad de
> reconstrucción**, no destreza predictiva frente a datos independientes.


In [ ]:
set.seed(2026)
puntos <- spatSample(borde, size = 70, method = "random")

datos <- data.frame(
  lon     = crds(puntos)[, 1],
  lat     = crds(puntos)[, 2],
  precip  = extract(precip, puntos)[, 2],   # [,2] porque extract() antepone ID
  altitud = extract(r_alt,  puntos)[, 2]
)
datos <- na.omit(datos)

# Proyección a km. El variograma mide semivarianza contra DISTANCIA:
#   1 grado de latitud  = 110.574 km (constante)
#   1 grado de longitud = 111.320 km * cos(latitud)
lat0 <- mean(datos$lat); lon0 <- mean(datos$lon)
datos$x_km <- (datos$lon - lon0) * 111.320 * cos(lat0 * pi / 180)
datos$y_km <- (datos$lat - lat0) * 110.574


**Resultado:** 70 estaciones, ningún `NA`, **2 415 pares** para el semivariograma
(el molde trabaja con 780).


---
# Paso 3 — Análisis exploratorio espacial

## 3.1 Distribución de la precipitación

```
Min  34.93   Q1  51.72   Mediana  64.17   Media  91.09   Q3 125.88   Max 189.01
Desviación estándar : 51.32 mm
Coeficiente de variación : 56.3 %
Media / mediana : 1.42
Shapiro-Wilk : W = 0.8156, p = 6.4e-08  ->  SE RECHAZA normalidad
```

**Interpretación.** La media es 42 % mayor que la mediana: asimetría fuerte a la
derecha. De Q1 a la mediana hay 12 mm; de la mediana a Q3 hay 62 mm — cinco veces
más. La cola derecha es la vertiente pacífica.

Esto importa porque el kriging es un predictor **lineal**, óptimo bajo normalidad.
Pero la pregunta correcta no es si `precip` es normal, sino si lo son **los
residuales** después de quitar la tendencia. Se resuelve en el Paso 4.


![Distribución](resultados/fig03_distribucion.png)

## 3.2 Mapa de posting

![Posting](resultados/fig04_posting.png)

Los símbolos grandes se concentran al oeste. Confirma visualmente el gradiente.


## 3.3 Relación con las covariables

```
Correlación de Pearson con precip:
  x_km    -0.890      ← la longitud sola explica 0.890² = 79 % de la varianza
  altitud -0.651      ← sospechosa, ver Paso 4
  y_km    -0.274
```


![Covariables](resultados/fig05_covariables.png)

**Interpretación.** La longitud domina. La altitud parece fuerte pero está confundida
con ella (la costa es baja *y* occidental). La latitud aporta poco por sí sola.


---
# Paso 4 — Modelo de tendencia

Kriging universal = **tendencia determinística + residuo espacialmente correlacionado**.
Aquí se estima la primera parte.

## 4.1 La confusión entre altitud y longitud

```
cor(altitud, precip) = -0.651    (simple)
cor(altitud, x_km)   =  0.755    (colinealidad)

Modelo A:  precip ~ x_km + y_km            R² = 0.8336   R²adj = 0.8286
Modelo B:  precip ~ x_km + y_km + altitud  R² = 0.8449   R²adj = 0.8379

  (Intercept)  79.4979   p = 0.0000
  x_km         -1.1610   p = 0.0000
  y_km         +0.3165   p = 0.0000   ← correlación simple era -0.274
  altitud      +0.0098   p = 0.0317   ← ¡signo invertido!

Test F anidado A vs B:  F = 4.8166,  p = 0.0317  →  la altitud se queda
```

**Qué significa.** La correlación simple decía *"más alto = más seco"*. Lo que decía
en realidad era *"más al este = más seco"*: la costa es baja **y** occidental, la
cordillera es alta **y** oriental. Al controlar la posición, el signo se invierte.

Es un caso de libro de **confusión (confounding)**.


### Visto en cuatro figuras

![Colinealidad](resultados/fig07a_colinealidad.png)

**(A) La causa.** `r = 0,755`. No hay puntos verdes altos ni rojos bajos — **no existe
costa alta ni valle a nivel del mar**. Por eso el modelo no puede separarlas del todo.


![Correlación simple](resultados/fig07b_correlacion_simple.png)

**(B) El engaño.** La recta baja, pero los colores delatan el truco: el extremo
izquierdo (altitud ≈ 0, lluvia 180 mm) es **todo verde**. La pendiente negativa la
produce un grupo que es bajo *y* occidental.


![Estratificado](resultados/fig07c_estratificado.png)

**(C) La realidad, más rica de lo esperado.**

| Banda | Pendiente | Física |
|---|---|---|
| 🟢 Oeste (n=24) | **−0,0582** | Chocó: el máximo de lluvia está **al nivel del mar**; la masa húmeda es somera y descarga abajo. Subiendo, seca |
| 🟠 Centro (n=23) | **+0,0167** | Ladera de la cordillera Occidental: ascenso orográfico clásico |
| 🔴 Este (n=23) | **+0,0050** | Valle del Cauca y ladera de la Central: mismo mecanismo, más débil |
| Agrupado (n=70) | −0,0369 | promedio engañoso |

> ⚠️ **Limitación que hay que declarar.** El efecto de la altitud **no es homogéneo en
> el espacio**: son dos regímenes físicos opuestos. Lo correcto sería una interacción
> `altitud × longitud`, pero **ningún script de clase usa interacciones** — los cuatro
> ejemplos son `lm(z ~ x + y)` pelado. Se mantiene la deriva lineal y la limitación
> queda documentada.


![Variable añadida](resultados/fig07d_variable_anyadida.png)

**(D) El efecto parcial.** Panel canónico para confusión. Ejes = residuales de
`precip` y de `altitud` después de quitarle a ambos la posición. La pendiente,
**+0,0098**, es *exactamente* el coeficiente del `lm()` múltiple.


## 4.2 Diagnóstico de residuales — valida todo el enfoque

```
precip cruda  ->  Shapiro W = 0.8156,  p = 6.4e-08   ->  NO normal
residuales    ->  Shapiro W = 0.9750,  p = 0.1739    ->  normal ✅
sd:  51.32 mm  ->  20.21 mm
```

**Contesta la pregunta del Paso 3: no hay que transformar la variable.** La asimetría
de `precip` era la tendencia, no la distribución. Quitada la tendencia, los residuales
son normales y el kriging queda justificado como predictor lineal óptimo.


![Residuales](resultados/fig06_residuales.png)

**Panel (B)** el Q-Q sigue la recta con colas ligeramente cortas — aceptable.

⚠️ **Panel (C) muestra un problema real.** Los residuales dibujan una **U**:

```
ajustado ~30   → residual +30
ajustado ~100  → residual -40
ajustado ~170  → residual +25
```

Curvatura sistemática: la tendencia lineal en `x_km` está mal especificada, la
relación real es curva.

**Decisión.** Ningún script de clase usa tendencia polinómica, así que agregar
`I(x_km^2)` sacaría del whitelist. Se mantiene lineal — y la U es *estructura espacial
que la tendencia determinística no capturó*, que es **exactamente lo que el kriging
debe absorber**. El enunciado pide "evaluar el papel de la correlación espacial":
esa U **es** la evidencia de que importa.


## 4.3 Las mismas relaciones, sobre el mapa

![Mapa bandas](resultados/fig08a_mapa_bandas.png)

Las tres bandas de longitud usadas en el análisis estratificado, sobre el territorio.


![Mapa altitud](resultados/fig08b_mapa_altitud.png)

Relieve con las 70 estaciones encima. Se ven las dos cordilleras y el valle entre ellas.


![Mapa residuales](resultados/fig08c_mapa_residuales.png)

**🔴 La figura más importante del EDA.** Los residuales están **agrupados en manchas**,
no dispersos al azar:

| Zona | Color | Lectura |
|---|---|---|
| Centro del valle (−76,7 · 3,5-3,8) | 🔵 grandes | el modelo **sobreestima** |
| Flanco oriental (−75,9 · 3,9-4,1) | 🔴 grandes | el modelo **subestima** |
| Ladera pacífica (−77,2 · 3,7-4,0) | 🔴 | subestima |
| Norte (−76,2 · 4,5-4,9) | 🔵 | sobreestima |

Es la **U del panel (C) vista en el espacio**. La tendencia lineal no puede curvarse,
así que aplana el fondo del valle hacia arriba y las laderas hacia abajo.

Anticipa el resultado del Paso 5: **el Moran sobre residuales será claramente positivo.**


![Obs vs tendencia](resultados/fig08d_mapa_obs_vs_tendencia.png)

Observado contra ajustado. El modelo reproduce el gradiente general pero suaviza los
extremos — visible en que los símbolos del panel (B) son más uniformes.


---
# Errores encontrados (y resueltos)

Vale la pena tenerlos a mano: tres de los cuatro fallan **en silencio**.

### 1. 🔴 Bug en `Script_Spatial_Analytics.R` del profesor, línea 77

```r
W <- matrix(ifelse(row_sums == 0, 0, W / row_sums), nrow = n, ncol = n)
```

`ifelse()` devuelve un objeto del largo del **test**, no del resultado. `row_sums`
mide `n`, así que devuelve `n` valores en vez de `n²`: se queda con la primera columna
de `W/row_sums` y `matrix()` la recicla. Cada columna queda idéntica y la
estandarización por filas **no ocurre**.

```
metodo del script:  suma por fila = 0 2 0 0   (debería ser 1 1 0 0)
con sweep():        suma por fila = 1 1 0 0   ✅
```

**Solución:** `sweep(W, 1, ifelse(filas == 0, 1, filas), "/")`.

No invalida el curso — la teoría del script está bien y es un *gotcha* clásico de R.
Pero copiado tal cual, el Moran sale mal.

### 2. 🔴 `as.vector(ext())` trae nombres

```r
e  <- as.vector(ext(borde))     # nombres: xmin xmax ymin ymax
lg <- c(x = e[1], y = e[4])     # el nombre queda "x.xmin", NO "x"
lg["x"]                         # -> NA
legend(NA, NA, ...)             # no dibuja nada y NO da error
```

**Solución:** `unname(as.vector(ext(borde)))`.

### 3. `terra::plot` recorta las leyendas

Deja el área de dibujo pegada al polígono, así que `legend("topleft")` se sale por
arriba. **Solución:** coordenadas explícitas vía `esquina_leyenda()`.

### 4. El error del denominador (Paso 1.3)

Medir cobertura contra el rectángulo en vez del departamento. Descartaba la mejor
covariable. **Solución:** definir la máscara antes de calcular porcentajes.


---
# Lo que sigue

| Paso | Qué | Bloque del enunciado | Estado |
|---|---|---|---|
| 1 | Auditoría de datos | — | ✅ |
| 2 | Borde y muestreo | — | ✅ |
| 3 | EDA espacial | **1. EDA** | ✅ |
| 4 | Tendencia y confusión | **2. Modelación** | ✅ |
| 5 | Moran's I y Geary's C a mano | **1. EDA** | ⏳ |
| 6 | Semivariograma: nube → bins → `optim()` | **2. Modelación** | — |
| 7 | Sistema kriging con `solve()` | **2. Modelación** | — |
| 8 | LOOCV y lectura honesta del R² | **3. Validación** | — |
| 9 | Mapa de predicción + incertidumbre | **4. Predicción** | — |
| 10 | Redactar el PDF (≤ 10 páginas) | — | — |

### Recordatorio de entrega

| | |
|---|---|
| **Plazo** | domingo **13-sep-2026, 23:59** — solo Moodle |
| **Documento** | PDF, máx. **10 páginas**, una sola columna |
| **Código** | **un único** `.R`, reproducible, en archivo aparte |
| **Prohibido** | R Markdown, solo-Colab, manuscrito |
| **Si se usó IA** | `.docx` aparte con los prompts y la herramienta |
| **Sustentación** | se sortea un integrante; ausencia sin excusa = **0.0** |
